# Cross-entropy method

Learn the cross-entropy method on discrete Taxi states, then use a neural-network policy for CartPole.

This exercise targets Python 3.12+:

```bash
python -m pip install "gymnasium[classic-control]>=1.3,<2" numpy matplotlib scikit-learn ipython
```


In [ ]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt

from IPython.display import clear_output

%matplotlib inline


In [ ]:
env = gym.make("Taxi-v4", render_mode="ansi")
observation, info = env.reset()
print(env.render())


In [ ]:
n_states = env.observation_space.n
n_actions = env.action_space.n

print("n_states=%i, n_actions=%i"%(n_states,n_actions))

# Create stochastic policy

This time our policy should be a probability distribution.

```policy[s,a] = P(take action a | in state s)```

Since we still use integer state and action representations, you can use a 2-dimensional array to represent the policy.

Please initialize policy __uniformly__, that is, probabililities of all actions should be equal.


# Initialize policy (0.5pts)

In [ ]:
policy = None  # TODO: create a uniform array of shape (n_states, n_actions)
assert isinstance(policy, np.ndarray)
assert np.allclose(policy, 1.0 / n_actions)
assert np.allclose(np.sum(policy, axis=1), 1.0)
print("Ok!")


# Play the game (0.5pts)

Just like before, but we also record all states and actions we took.

In [ ]:
def generate_session(policy, t_max=10**4):
    """Return a Taxi episode's states, actions, and total reward."""
    states, actions = [], []
    total_reward = 0.0
    state, info = env.reset()

    for _ in range(t_max):
        action = np.random.choice(n_actions, p=policy[state])
        next_state, reward, terminated, truncated, info = env.step(action)

        # TODO: record state, action, and reward.

        state = next_state
        if terminated or truncated:
            break

    return states, actions, total_reward


In [ ]:
states, actions, reward = generate_session(policy)
assert isinstance(states, list) and isinstance(actions, list)
assert len(states) == len(actions)
assert isinstance(reward, (float, np.floating))


In [ ]:
#let's see the initial reward distribution
import matplotlib.pyplot as plt
%matplotlib inline

sample_rewards = [generate_session(policy,t_max=1000)[-1] for _ in range(200)]

plt.hist(sample_rewards,bins=20);
plt.vlines([np.percentile(sample_rewards,50)],[0],[100],label="50'th percentile",color='green')
plt.vlines([np.percentile(sample_rewards,90)],[0],[100],label="90'th percentile",color='red')
plt.legend()

### Crossentropy method steps (1pts)

In [ ]:
def select_elites(states_batch, actions_batch, rewards_batch, percentile=50):
    """Select state-action pairs from sessions at or above the reward percentile.

    Preserve session order and the order of steps within each session.
    States may be integers (Taxi) or vectors (CartPole).
    """
    reward_threshold = None  # TODO: compute with np.percentile
    elite_session_indices = []  # TODO: select sessions with reward >= threshold

    # TODO: combine states and actions from selected sessions in order.
    # For Taxi, a flat state array is sufficient. For CartPole, stack rows.
    elite_states = None
    elite_actions = None
    return elite_states, elite_actions


In [ ]:
states_batch = [
    [1, 2, 3],       # game 1
    [4, 2, 0, 2],    # game 2
    [3, 1],          # game 3
]
actions_batch = [
    [0, 2, 4],       # game 1
    [3, 2, 0, 1],    # game 2
    [3, 3],          # game 3
]
rewards_batch = np.array([3, 4, 5])

test_result_0 = select_elites(states_batch, actions_batch, rewards_batch, percentile=0)
test_result_30 = select_elites(states_batch, actions_batch, rewards_batch, percentile=30)
test_result_90 = select_elites(states_batch, actions_batch, rewards_batch, percentile=90)
test_result_100 = select_elites(states_batch, actions_batch, rewards_batch, percentile=100)

assert np.array_equal(test_result_0[0], [1, 2, 3, 4, 2, 0, 2, 3, 1])
assert np.array_equal(test_result_0[1], [0, 2, 4, 3, 2, 0, 1, 3, 3])
assert np.array_equal(test_result_30[0], [4, 2, 0, 2, 3, 1])
assert np.array_equal(test_result_30[1], [3, 2, 0, 1, 3, 3])
assert np.array_equal(test_result_90[0], [3, 1])
assert np.array_equal(test_result_90[1], [3, 3])
assert np.array_equal(test_result_100[0], [3, 1])
assert np.array_equal(test_result_100[1], [3, 3])
print("Ok!")


In [ ]:
def update_policy(elite_states,elite_actions):
    """
    Given old policy and a list of elite states/actions from select_elites,
    return new updated policy where each action probability is proportional to
    
    policy[s_i,a_i] ~ #[occurences of si and ai in elite states/actions]
    
    Don't forget to normalize policy to get valid probabilities and handle 0/0 case.
    In case you never visited a state, set probabilities for all actions to 1./n_actions
    
    :param elite_states: 1D list of states from elite sessions
    :param elite_actions: 1D list of actions from elite sessions
    
    """
    
    new_policy = np.zeros([n_states,n_actions])
    
    #<Your code here: update probabilities for actions given elite states & actions>
    #Don't forget to set 1/n_actions for all actions in unvisited states.
    
    return new_policy

In [ ]:

elite_states, elite_actions = ([1, 2, 3, 4, 2, 0, 2, 3, 1], [0, 2, 4, 3, 2, 0, 1, 3, 3])


new_policy = update_policy(elite_states,elite_actions)

assert np.isfinite(new_policy).all(), "Your new policy contains NaNs or +-inf. Make sure you don't divide by zero."
assert np.all(new_policy>=0), "Your new policy can't have negative action probabilities"
assert np.allclose(new_policy.sum(axis=-1),1), "Your new policy should be a valid probability distribution over actions"
reference_answer = np.array([
       [ 1.        ,  0.        ,  0.        ,  0.        ,  0.        ],
       [ 0.5       ,  0.        ,  0.        ,  0.5       ,  0.        ],
       [ 0.        ,  0.33333333,  0.66666667,  0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        ,  0.5       ,  0.5       ]])
print(new_policy[:4,:5])
assert np.allclose(new_policy[:4,:5],reference_answer)
print("Ok!")

# Training loop (1pts)
Generate sessions, select N best and fit to those.

In [ ]:
def show_progress(rewards_batch, log, percentile, reward_range=None):
    """Display the mean reward and elite threshold during training."""
    mean_reward = np.mean(rewards_batch)
    threshold = np.percentile(rewards_batch, percentile)
    log.append((mean_reward, threshold))

    clear_output(wait=True)
    print(f"mean reward = {mean_reward:.3f}, threshold = {threshold:.3f}")
    fig, axes = plt.subplots(1, 2, figsize=(9, 4))
    axes[0].plot([row[0] for row in log], label="Mean reward")
    axes[0].plot([row[1] for row in log], label="Elite threshold")
    axes[0].set_xlabel("Iteration")
    axes[0].set_ylabel("Reward")
    axes[0].legend()
    axes[0].grid(True)

    axes[1].hist(rewards_batch, bins=20, range=reward_range)
    axes[1].axvline(threshold, label="Elite threshold", color="red")
    axes[1].set_xlabel("Episode reward")
    axes[1].set_ylabel("Count")
    axes[1].legend()
    axes[1].grid(True)
    plt.tight_layout()
    plt.show()


In [ ]:
# Start the training exercise with a uniform Taxi policy.
policy = np.full((n_states, n_actions), 1.0 / n_actions)


In [ ]:
n_sessions = 250
percentile = 70
learning_rate = 0.5
log = []

for iteration in range(100):
    sessions = [generate_session(policy) for _ in range(n_sessions)]
    states_batch, actions_batch, rewards_batch = zip(*sessions)
    elite_states, elite_actions = select_elites(
        states_batch, actions_batch, rewards_batch, percentile=percentile
    )
    new_policy = update_policy(elite_states, elite_actions)
    policy = None  # TODO: combine old and new policies using learning_rate
    show_progress(rewards_batch, log, percentile)


# Tabular crossentropy method

You may have noticed that the taxi problem quickly converges from -100 to a near-optimal score and then descends back into -50/-100. This is in part because the environment has some innate randomness. Namely, the starting points of passenger/driver change from episode to episode.

### Tasks
- __1.1__ (1 pts) Find out how the algorithm performance changes if you change different percentile and different n_samples. - Build graph with different percentiles on the same graph. And second one for sample size
- __1.2__ (1 pts) Tune the algorithm to end up with positive average score.

It's okay to modify the existing code.


In [ ]:
env.close()
env = gym.make("Taxi-v4")
state, info = env.reset()
n_states = env.observation_space.n
n_actions = env.action_space.n
policy = np.full((n_states, n_actions), 1.0 / n_actions)

assert isinstance(policy, np.ndarray)
assert np.allclose(policy.sum(axis=1), 1.0)
states, actions, reward = generate_session(policy)
assert isinstance(states, list) and isinstance(actions, list)
assert len(states) == len(actions)
assert isinstance(reward, (float, np.floating))
step_counter = 40


In [ ]:
n_sessions = 250
percentiles = []  # TODO: choose percentiles to compare
learning_rate = 0.5
mean_rewards_by_percentile = []

for percentile in percentiles:
    policy = np.full((n_states, n_actions), 1.0 / n_actions)
    current_means = []
    for iteration in range(step_counter):
        sessions = [generate_session(policy) for _ in range(n_sessions)]
        states_batch, actions_batch, rewards_batch = zip(*sessions)
        elite_states, elite_actions = select_elites(
            states_batch, actions_batch, rewards_batch, percentile=percentile
        )
        # TODO: update policy and record the mean reward.
    mean_rewards_by_percentile.append(current_means)


In [ ]:
# how do different percentiles affect training efficiency?

In [ ]:
session_counts = []  # TODO: choose batch sizes to compare
percentile = 70
learning_rate = 0.5
mean_rewards_by_session_count = []

for n_sessions in session_counts:
    policy = np.full((n_states, n_actions), 1.0 / n_actions)
    current_means = []
    for iteration in range(step_counter):
        sessions = [generate_session(policy) for _ in range(n_sessions)]
        states_batch, actions_batch, rewards_batch = zip(*sessions)
        elite_states, elite_actions = select_elites(
            states_batch, actions_batch, rewards_batch, percentile=percentile
        )
        # TODO: update policy and record the mean reward.
    mean_rewards_by_session_count.append(current_means)


# Stabilize positive rewards by averaging policy across 10 games (2 pts)

In [ ]:
def generate_session(policy, t_max=10**4):
    """Return a Taxi episode's states, actions, and total reward.

    TODO: extend the earlier function for the policy-averaging exercise.
    Use `state, info = env.reset()` and unpack each step as
    `next_state, reward, terminated, truncated, info = env.step(action)`.
    Stop on `terminated or truncated`.
    """
    raise NotImplementedError("Complete the policy-averaging session generator")


In [ ]:
n_sessions = 75
percentile = 70
learning_rate = 0.2
log = []
mean = []
policy = np.full((n_states, n_actions), 1.0 / n_actions)

for iteration in range(1000):
    sessions = [generate_session(policy) for _ in range(n_sessions)]
    states_batch, actions_batch, rewards_batch = zip(*sessions)
    elite_states, elite_actions = select_elites(
        states_batch, actions_batch, rewards_batch, percentile=percentile
    )
    # TODO: average policies across games and update policy.
    show_progress(rewards_batch, log, percentile)
    if np.mean(rewards_batch) > 10:
        print("win!")
        break


Количество иттераций для достжения положительного результата возросло, но модель не только достигает большей положительной награды, но и стабильно сохраняет его.

# Digging deeper: approximate crossentropy with neural nets (2 pts)

In this section we will train a neural network policy for continuous state space game

In [ ]:
env.close()
env = gym.make("CartPole-v1", render_mode="rgb_array")
initial_observation, info = env.reset()
n_actions = env.action_space.n
plt.imshow(env.render())
plt.axis("off")


In [ ]:
from sklearn.neural_network import MLPClassifier

agent = MLPClassifier(
    hidden_layer_sizes=(20, 20),
    activation="tanh",
    warm_start=True,
    max_iter=1,
)
initial_observation, info = env.reset()
agent.fit(
    np.repeat(initial_observation[None, :], n_actions, axis=0),
    np.arange(n_actions),
)


In [ ]:
def generate_session(t_max=1000):
    """Return a CartPole episode's states, actions, and total reward."""
    states, actions = [], []
    total_reward = 0.0
    state, info = env.reset()

    for _ in range(t_max):
        probs = None  # TODO: obtain action probabilities with agent.predict_proba
        action = np.random.choice(n_actions, p=probs)
        next_state, reward, terminated, truncated, info = env.step(action)
        # TODO: record state, action, and reward.
        state = next_state
        if terminated or truncated:
            break

    return states, actions, total_reward


In [ ]:
def select_elites(states_batch, actions_batch, rewards_batch, percentile=50):
    """Select elite sessions without constructing ragged NumPy arrays."""
    threshold = np.percentile(rewards_batch, percentile)
    selected = [
        index for index, reward in enumerate(rewards_batch)
        if reward >= threshold
    ]
    elite_states = np.concatenate(
        [np.asarray(states_batch[index]) for index in selected], axis=0
    )
    elite_actions = np.concatenate(
        [np.asarray(actions_batch[index]) for index in selected], axis=0
    )
    return elite_states, elite_actions


In [ ]:
n_sessions = 100
percentile = 70
log = []

for iteration in range(100):
    sessions = [generate_session() for _ in range(n_sessions)]
    states_batch, actions_batch, rewards_batch = zip(*sessions)
    elite_states, elite_actions = select_elites(
        states_batch, actions_batch, rewards_batch, percentile=percentile
    )
    # TODO: fit agent to the elite states and actions.
    show_progress(rewards_batch, log, percentile, reward_range=(0, 500))
    if np.mean(rewards_batch) > 190:
        print("You win! You may stop training now.")
        break


# Report (1 pts)

In [ ]:
# Describe what you did here.  Preferably with plot/report to support it